## 1. Imports

In [1]:
import pandas as pd
import numpy as np
import joblib
import os
import warnings
warnings.filterwarnings('ignore')

from lightgbm import LGBMClassifier, early_stopping
import xgboost as xgb
from sklearn.metrics import average_precision_score

## 2. Load Data

In [2]:
train_transaction = pd.read_csv("../data/train_transaction.csv")
train_identity    = pd.read_csv("../data/train_identity.csv")

print("train_transaction:", train_transaction.shape)
print("train_identity:   ", train_identity.shape)

train_transaction: (590540, 394)
train_identity:    (144233, 41)


## 3. Merge & Time-Based Train/Val Split

In [3]:
# Merge transaction + identity
train = train_transaction.merge(train_identity, on='TransactionID', how='left')
print("Merged shape:", train.shape)

# 80/20 time-based split (no data leakage)
threshold = train['TransactionDT'].quantile(0.80)
print(f"Threshold: {threshold}")
print(f"Train fraud rate: {train[train['TransactionDT'] < threshold]['isFraud'].mean():.4f}")
print(f"Val   fraud rate: {train[train['TransactionDT'] >= threshold]['isFraud'].mean():.4f}")

Merged shape: (590540, 434)
Threshold: 12192853.600000001
Train fraud rate: 0.0351
Val   fraud rate: 0.0344


## 4. Preprocessing

In [4]:
# M1-M3,M5,M7-M9: convert T/F strings to 0/1
m_bool_cols = ['M1','M2','M3','M5','M7','M8','M9']
for col in m_bool_cols:
    train[col] = train[col].map({'T': 1, 'F': 0})

# Categorical columns: encode as pandas category on full train df
# (slice AFTER so X_train/X_val inherit the same categories)
cat_cols = [
    'ProductCD', 'card4', 'card6',
    'DeviceType', 'id_30', 'id_31',
    'P_emaildomain', 'R_emaildomain',
    'M4', 'M6'
]
for col in cat_cols:
    train[col] = train[col].astype('category')

print("Preprocessing done.")

Preprocessing done.


## 5. Feature Engineering
All engineered features are computed on the full `train` df **before** slicing, except uid aggregation stats which must be computed on X_train only to avoid leakage.

In [5]:
# DeviceBrand: first token of DeviceInfo
train['DeviceBrand'] = (
    train['DeviceInfo']
    .fillna('missing')
    .str.split()
    .str[0]
)
train['DeviceBrand'] = train['DeviceBrand'].astype('category')
print("DeviceBrand unique values:", train['DeviceBrand'].nunique())

DeviceBrand unique values: 1183


In [6]:
# Time-based split — done AFTER all train-level preprocessing
X_train = train[train['TransactionDT'] < threshold].copy()
X_val   = train[train['TransactionDT'] >= threshold].copy()

print(f"X_train: {X_train.shape}  |  X_val: {X_val.shape}")

X_train: (472432, 435)  |  X_val: (118108, 435)


In [7]:
# ── uid1: card1 + addr1 ────────────────────────────────────────────────────────
X_train['uid'] = X_train['card1'].astype(str) + '_' + X_train['addr1'].astype(str)
X_val['uid']   = X_val['card1'].astype(str)   + '_' + X_val['addr1'].astype(str)

# uid_count
uid_count_map = X_train.groupby('uid').size()
X_train['uid_count'] = X_train['uid'].map(uid_count_map)
X_val['uid_count']   = X_val['uid'].map(uid_count_map).fillna(0)

# uid_amt_mean
uid_amt_mean_map = X_train.groupby('uid')['TransactionAmt'].mean()
X_train['uid_amt_mean'] = X_train['uid'].map(uid_amt_mean_map)
X_val['uid_amt_mean']   = X_val['uid'].map(uid_amt_mean_map).fillna(X_train['TransactionAmt'].mean())

# uid_amt_std
uid_amt_std_map = X_train.groupby('uid')['TransactionAmt'].std()
X_train['uid_amt_std'] = X_train['uid'].map(uid_amt_std_map).fillna(0)
X_val['uid_amt_std']   = X_val['uid'].map(uid_amt_std_map).fillna(0)

# uid_C1/C2/C13/C14 means
for col in ['C1', 'C2', 'C13', 'C14']:
    mapping = X_train.groupby('uid')[col].mean()
    X_train[f'uid_{col}_mean'] = X_train['uid'].map(mapping)
    X_val[f'uid_{col}_mean']   = X_val['uid'].map(mapping).fillna(X_train[col].mean())

print("uid1 features done.")

uid1 features done.


In [8]:
# ── D1n: normalised D1 day ──────────────────────────────────────────────────────
for df in [X_train, X_val]:
    df['D1n'] = np.floor(df['TransactionDT'] / (24*60*60)) - df['D1']

print("D1n done.")

D1n done.


In [9]:
# ── uid2: card1+card2+card3+card5 ───────────────────────────────────────────────
for df in [X_train, X_val]:
    df['uid2'] = (
        df['card1'].astype(str) + '_' +
        df['card2'].astype(str) + '_' +
        df['card3'].astype(str) + '_' +
        df['card5'].astype(str)
    )

uid2_amt_mean_map = X_train.groupby('uid2')['TransactionAmt'].mean()
X_train['uid2_amt_mean'] = X_train['uid2'].map(uid2_amt_mean_map)
X_val['uid2_amt_mean']   = X_val['uid2'].map(uid2_amt_mean_map).fillna(X_train['TransactionAmt'].mean())

uid2_amt_std_map = X_train.groupby('uid2')['TransactionAmt'].std()
X_train['uid2_amt_std'] = X_train['uid2'].map(uid2_amt_std_map).fillna(0)
X_val['uid2_amt_std']   = X_val['uid2'].map(uid2_amt_std_map).fillna(0)

print("uid2 features done.")

uid2 features done.


In [10]:
# ── uid_d1: card1 + D1 normalised day ───────────────────────────────────────────
for df in [X_train, X_val]:
    df['uid_d1'] = (
        df['card1'].astype(str) + '_' +
        np.floor(df['TransactionDT'] / (24*60*60) - df['D1']).astype(str)
    )

uid_d1_amt_mean_map = X_train.groupby('uid_d1')['TransactionAmt'].mean()
X_train['uid_d1_amt_mean'] = X_train['uid_d1'].map(uid_d1_amt_mean_map)
X_val['uid_d1_amt_mean']   = X_val['uid_d1'].map(uid_d1_amt_mean_map).fillna(X_train['TransactionAmt'].mean())

print("uid_d1 features done.")

uid_d1 features done.


## 6. Final Immutable Feature List

In [11]:
features = [
    'TransactionAmt',
    'ProductCD',
    'card4',
    'card6',
    'DeviceType',
    'id_30',
    'id_31',
    'P_emaildomain',
    'R_emaildomain',
    'addr1',
    'addr2',
    'M4',
    'M6',
    'V188',
    'V189',
    'V200',
    'V201',
    'V242',
    'V244',
    'V246',
    'V257',
    'V258',
    'V147',
    'V148',
    'V149',
    'V154',
    'V155',
    'V156',
    'V157',
    'V158',
    'D1',
    'D2',
    'D3',
    'D4',
    'D5',
    'D10',
    'D11',
    'D15',
    'C1',
    'C2',
    'C3',
    'C4',
    'C5',
    'C6',
    'C7',
    'C8',
    'C9',
    'C10',
    'C11',
    'C12',
    'C13',
    'C14',
    'M1',
    'M2',
    'M3',
    'M5',
    'M7',
    'M8',
    'M9',
    'V39',
    'V40',
    'V42',
    'V43',
    'V44',
    'V45',
    'V170',
    'V171',
    'V46',
    'V47',
    'V48',
    'V49',
    'V50',
    'V51',
    'V52',
    'id_01',
    'id_02',
    'id_03',
    'id_04',
    'id_05',
    'id_06',
    'id_09',
    'id_10',
    'id_11',
    'id_13',
    'id_17',
    'id_20',
    'DeviceBrand',
    'uid_count',
    'uid_C1_mean',
    'uid_C2_mean',
    'uid_C13_mean',
    'uid_C14_mean',
    'uid_amt_mean',
    'uid_amt_std',
    'D1n',
    'uid2_amt_mean',
    'uid2_amt_std',
    'uid_d1_amt_mean',
]

# Verify uniqueness
assert len(features) == len(set(features)), "Duplicate features found!"

# Verify all features exist in X_train
missing = [f for f in features if f not in X_train.columns]
assert len(missing) == 0, f"Features missing from X_train: {missing}"

print(f"Feature count: {len(features)} — all present in X_train ✓")

Feature count: 98 — all present in X_train ✓


## 7. Categorical Dtype Alignment
Force X_val to use exactly the same category levels as X_train for every categorical column. This prevents the LightGBM `categorical_feature do not match` error.

In [12]:
for col in features:
    if str(X_train[col].dtype) == 'category':
        X_val[col] = pd.Categorical(
            X_val[col],
            categories=X_train[col].cat.categories
        )

print("Categorical alignment done.")
cat_cols_in_features = [c for c in features if str(X_train[c].dtype) == 'category']
print("Categorical features:", cat_cols_in_features)

Categorical alignment done.
Categorical features: ['ProductCD', 'card4', 'card6', 'DeviceType', 'id_30', 'id_31', 'P_emaildomain', 'R_emaildomain', 'M4', 'M6', 'DeviceBrand']


## 8. Train LightGBM (Final Config)

In [13]:
model = LGBMClassifier(
    n_estimators=3000,
    learning_rate=0.01,
    num_leaves=256,
    subsample=0.7,
    colsample_bytree=0.7,
    random_state=42,
    n_jobs=-1
)

model.fit(
    X_train[features],
    X_train['isFraud'],
    eval_set=[(X_val[features], X_val['isFraud'])],
    eval_metric='average_precision',
    callbacks=[early_stopping(100)]
)

print(f"LGB best iteration: {model.best_iteration_}")

[LightGBM] [Info] Number of positive: 16599, number of negative: 455833
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.030225 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 10586
[LightGBM] [Info] Number of data points in the train set: 472432, number of used features: 98
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.035135 -> initscore=-3.312784
[LightGBM] [Info] Start training from score -3.312784
Training until validation scores don't improve for 100 rounds
Early stopping, best iteration is:
[1065]	valid_0's average_precision: 0.62005	valid_0's binary_logloss: 0.0786771
LGB best iteration: 1065


## 9. Train XGBoost (Final Config)

In [16]:
xgb_model = xgb.XGBClassifier(
    n_estimators=2000,
    learning_rate=0.02,
    max_depth=12,
    min_child_weight=5,
    subsample=0.8,
    colsample_bytree=0.4,
    eval_metric="aucpr",
    early_stopping_rounds=100,
    random_state=42,
    tree_method="hist",
    enable_categorical=True
)

xgb_model.fit(
    X_train[features],
    X_train['isFraud'],
    eval_set=[(X_val[features], X_val['isFraud'])],
    verbose=100
)

print(f"XGB best iteration: {xgb_model.best_iteration}")

[0]	validation_0-aucpr:0.35100
[100]	validation_0-aucpr:0.51210
[200]	validation_0-aucpr:0.55062
[300]	validation_0-aucpr:0.57157


KeyboardInterrupt: 

In [29]:
import joblib

xgb_model = joblib.load("xgb_best.pkl")

In [30]:
print(xgb_model.get_booster().num_boosted_rounds())

1500


In [31]:
xgb_preds = xgb_model.predict_proba(X_val[features])[:, 1]

ValueError: feature_names mismatch: ['TransactionAmt', 'ProductCD', 'card4', 'card6', 'DeviceType', 'id_30', 'id_31', 'P_emaildomain', 'R_emaildomain', 'addr1', 'addr2', 'M4', 'M6', 'V188', 'V189', 'V200', 'V201', 'V242', 'V244', 'V246', 'V257', 'V258', 'V147', 'V148', 'V149', 'V154', 'V155', 'V156', 'V157', 'V158', 'D1', 'D2', 'D3', 'D4', 'D5', 'D10', 'D11', 'D15', 'C1', 'C2', 'C3', 'C4', 'C5', 'C6', 'C7', 'C8', 'C9', 'C10', 'C11', 'C12', 'C13', 'C14', 'M1', 'M2', 'M3', 'M5', 'M7', 'M8', 'M9', 'V39', 'V40', 'V42', 'V43', 'V44', 'V45', 'V170', 'V171', 'V46', 'V47', 'V48', 'V49', 'V50', 'V51', 'V52', 'id_01', 'id_02', 'id_03', 'id_04', 'id_05', 'id_06', 'id_09', 'id_10', 'id_11', 'id_13', 'id_17', 'id_20', 'D6', 'D7', 'D8', 'D9', 'D12', 'D13', 'D14', 'DeviceBrand', 'uid_count', 'uid_C1_mean', 'uid_C2_mean', 'uid_C13_mean', 'uid_C14_mean', 'uid_amt_mean', 'uid_amt_std', 'uid_amt_max', 'D1n', 'uid2_amt_mean', 'uid2_amt_std', 'uid_d1_amt_mean'] ['TransactionAmt', 'ProductCD', 'card4', 'car

## 10. Ensemble & Evaluate

In [46]:
lgb_preds = model.predict_proba(X_val[lgb_features])[:, 1]
xgb_preds = xgb_model.predict_proba(X_val[features])[:, 1]

# Best weight from grid search in original notebook: 0.59 XGB / 0.41 LGB
ensemble_preds = 0.59 * xgb_preds + 0.41 * lgb_preds

lgb_prauc      = average_precision_score(X_val['isFraud'], lgb_preds)
xgb_prauc      = average_precision_score(X_val['isFraud'], xgb_preds)
ensemble_prauc = average_precision_score(X_val['isFraud'], ensemble_preds)

print(f"LightGBM  PR-AUC : {lgb_prauc:.6f}")
print(f"XGBoost   PR-AUC : {xgb_prauc:.6f}")
print(f"Ensemble  PR-AUC : {ensemble_prauc:.6f}  (0.59*XGB + 0.41*LGB)")

LightGBM  PR-AUC : 0.620050
XGBoost   PR-AUC : 0.597070
Ensemble  PR-AUC : 0.614582  (0.59*XGB + 0.41*LGB)


## 11. Save Models

In [ ]:
joblib.dump(model,     'lgb_final.pkl')
joblib.dump(xgb_model, 'xgb_final.pkl')
joblib.dump(features,  'features_final.pkl')

print("Saved:")
print("  lgb_final.pkl      :", os.path.exists('lgb_final.pkl'))
print("  xgb_final.pkl      :", os.path.exists('xgb_final.pkl'))
print("  features_final.pkl :", os.path.exists('features_final.pkl'))

In [18]:
import joblib

xgb_model = joblib.load("xgb_634.pkl")

In [24]:
print(type(xgb_model))
print(hasattr(xgb_model, "_Booster"))

try:
    print(xgb_model.get_booster())
except Exception as e:
    print(e)

<class 'xgboost.sklearn.XGBClassifier'>
True


In [28]:
import os

print(os.path.getsize("xgb_best.pkl"))

14738472


In [26]:
import os

print(os.getcwd())


/Users/shamsundar/Projects/Main Major project/fraud-detection-engine/notebooks


In [27]:
import os

for f in os.listdir():
    if f.endswith(".pkl"):
        print(f, os.path.getsize(f))

xgb_best.pkl 14738472
features_634.pkl 828
features_best.pkl 828
lgb_best.pkl 3798244
lgb_634.pkl 25868244
xgb_634.pkl 790


In [32]:
features = list(xgb_model.get_booster().feature_names)

In [33]:
print(len(features))
print(features[-15:])   # inspect the tail

106
['D13', 'D14', 'DeviceBrand', 'uid_count', 'uid_C1_mean', 'uid_C2_mean', 'uid_C13_mean', 'uid_C14_mean', 'uid_amt_mean', 'uid_amt_std', 'uid_amt_max', 'D1n', 'uid2_amt_mean', 'uid2_amt_std', 'uid_d1_amt_mean']


In [37]:
xgb_preds = xgb_model.predict_proba(X_val[features])[:, 1]

In [35]:
missing = [f for f in xgb_model.get_booster().feature_names if f not in X_val.columns]
print(missing)

['uid_amt_max']


In [36]:
uid_amt_max_map = X_train.groupby("uid")["TransactionAmt"].max()

X_train["uid_amt_max"] = X_train["uid"].map(uid_amt_max_map)
X_val["uid_amt_max"] = X_val["uid"].map(uid_amt_max_map)

In [39]:
print("LGB expects:", len(model.feature_name_))
print(model.feature_name_)

LGB expects: 98
['TransactionAmt', 'ProductCD', 'card4', 'card6', 'DeviceType', 'id_30', 'id_31', 'P_emaildomain', 'R_emaildomain', 'addr1', 'addr2', 'M4', 'M6', 'V188', 'V189', 'V200', 'V201', 'V242', 'V244', 'V246', 'V257', 'V258', 'V147', 'V148', 'V149', 'V154', 'V155', 'V156', 'V157', 'V158', 'D1', 'D2', 'D3', 'D4', 'D5', 'D10', 'D11', 'D15', 'C1', 'C2', 'C3', 'C4', 'C5', 'C6', 'C7', 'C8', 'C9', 'C10', 'C11', 'C12', 'C13', 'C14', 'M1', 'M2', 'M3', 'M5', 'M7', 'M8', 'M9', 'V39', 'V40', 'V42', 'V43', 'V44', 'V45', 'V170', 'V171', 'V46', 'V47', 'V48', 'V49', 'V50', 'V51', 'V52', 'id_01', 'id_02', 'id_03', 'id_04', 'id_05', 'id_06', 'id_09', 'id_10', 'id_11', 'id_13', 'id_17', 'id_20', 'DeviceBrand', 'uid_count', 'uid_C1_mean', 'uid_C2_mean', 'uid_C13_mean', 'uid_C14_mean', 'uid_amt_mean', 'uid_amt_std', 'D1n', 'uid2_amt_mean', 'uid2_amt_std', 'uid_d1_amt_mean']


In [40]:
missing = set(model.feature_name_) - set(features)
extra = set(features) - set(model.feature_name_)

print("Missing:", missing)
print("Extra:", extra)

Missing: set()
Extra: {'D13', 'D9', 'D12', 'D8', 'uid_amt_max', 'D14', 'D7', 'D6'}


In [41]:
lgb_features = model.feature_name_

In [42]:
print(len(lgb_features))
print(len(features))

98
106


In [43]:
lgb_preds = model.predict_proba(
    X_val[lgb_features]
)[:, 1]